# NB07 · Organic Structures, Isomers, and Functional Groups

**Part B: Data & Chemistry** · Estimated time: ~75 min · Difficulty: ●●○

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/notebooks/nb07_organic_structures_and_functional_groups.ipynb)

---

### One-Line Summary
Understand why carbon forms the basis of vast chemical diversity, master constitutional and stereoisomerism, enumerate alkane isomers, recognize functional groups with SMARTS, compare Hückel's $4n+2$ rule with computational aromaticity models, and build functional-group fingerprints.

## 1. Learning Objectives

By the end of this notebook, you will be able to:
1. **Explain Organic Diversity**: Understand the hybridization, catenation, and tetrahedral $sp^3$ geometry ($109.5^\circ$) that generate vast organic chemical space.
2. **Differentiate Isomer Types**: Distinguish constitutional isomers from stereoisomers and understand the combinatorial growth of alkane isomers.
3. **Enumerate & Visualize Isomers**: Generate and depict all 5 constitutional isomers of hexane ($C_6H_{14}$) and analyze their physical property trends.
4. **Query Functional Groups via SMARTS**: Utilize `chemlib.descriptors.FUNCTIONAL_GROUPS_SMARTS` to identify and count chemically distinct functional groups.
5. **Evaluate Aromaticity Perception**: Apply Hückel's $(4n+2)$ rule to cyclic conjugated systems and compare manual evaluation with RDKit's computational aromaticity perception.
6. **Construct Functional Group Fingerprints**: Generate binary presence/absence vectors from functional group profiles and compute pairwise Tanimoto similarities.
7. **Profile P-gp Substrates**: Preview multi-feature functional group profiling on a curated drug transport dataset.

## 2. Prerequisites & Lecture Links

- **Prerequisites**: SMILES specification (`nb01`, `nb03`), molecular graphs (`nb04`), and chemical file formats (`nb05`).
- **Lecture Aligned**: Lecture Deck 04 (*Organic Structures, Isomers & Functional Groups*).
- **Key Concepts**: $sp^3$ carbon, constitutional isomers, Cayley alkane enumeration, SMARTS recursive patterns, Hückel aromaticity ($4n+2$), Checkmol, functional-group fingerprinting.

In [ ]:
# Setup and environment configuration
import os, sys, random, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolTransforms

# Ensure src directory is in sys.path
repo_root = Path.cwd().parent if Path.cwd().name in ["notebooks", "solutions"] else Path.cwd()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import chemlib
from chemlib import descriptors, fingerprints

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

sns.set_theme(style="whitegrid", font_scale=1.1)
print("Environment initialized successfully.")
# Course reproducibility seeds and execution mode
import random
FULL_RUN = False


## 3. Organic Chemical Space and Carbon Versatility

The universe of chemical compounds is overwhelmingly organic:
- Historically, databases registered over **16 million organic compounds** compared to roughly **0.6 million inorganic compounds** (a >25:1 ratio; in modern PubChem, organic compounds exceed 115 million).
- **Why is Carbon Unique?**
  1. **Catenation**: Carbon readily forms stable covalent bonds with other carbon atoms, producing linear, branched, cyclic, and caged architectures.
  2. **Tetracovalence & Hybridization**: Carbon adopts $sp^3$ (tetrahedral, $109.5^\circ$), $sp^2$ (trigonal planar, $120^\circ$), and $sp$ (linear, $180^\circ$) geometries.
  3. **Heteroatom Compatibility**: Forms strong, polarizable bonds with H, O, N, S, P, and halogens.

Let's generate 3D coordinates for methane ($CH_4$) and verify its tetrahedral bond angle ($109.5^\circ$).

In [ ]:
# Build 3D methane with explicit hydrogens and measure H-C-H bond angle
methane = Chem.MolFromSmiles("C")
methane_3d = Chem.AddHs(methane)
AllChem.EmbedMolecule(methane_3d, randomSeed=SEED)
AllChem.MMFFOptimizeMolecule(methane_3d)

conf = methane_3d.GetConformer()
# In methane: C is atom 0, H atoms are 1, 2, 3, 4
angle_1 = rdMolTransforms.GetAngleDeg(conf, 1, 0, 2)
angle_2 = rdMolTransforms.GetAngleDeg(conf, 2, 0, 3)

print(f"Methane H1-C0-H2 bond angle: {angle_1:.2f}°")
print(f"Methane H2-C0-H3 bond angle: {angle_2:.2f}°")
print(f"Theoretical ideal tetrahedral angle: 109.47°")
assert abs(angle_1 - 109.47) < 1.0, "Angle should closely match ideal sp3 angle"

## 4. Isomerism: Constitutional vs. Stereoisomers

**Isomers** are distinct chemical compounds that share the identical molecular formula:

```
                            ISOMERS
                  (Same molecular formula)
                             │
              ┌──────────────┴──────────────┐
              ▼                             ▼
    CONSTITUTIONAL ISOMERS            STEREOISOMERS
 (Different atom connectivity)     (Same connectivity, different
   • Skeletal (chain branching)     spatial 3D orientation)
   • Positional (substituent locus) ┌───────────┴───────────┐
   • Functional group               ▼                       ▼
                              ENANTIOMERS             DIASTEREOMERS
                          (Non-superimposable     (Not mirror images;
                             mirror images)        cis/trans, meso)
```

### The Combinatorial Explosion of Alkane Isomers

> [!NOTE]
> **Lecture Errata / Clarification (E13)**: The classical table of alkane isomer counts (e.g., $C_7=9, C_{20}=366,319, C_{40} \approx 6.25 \times 10^{13}$) enumerated by Arthur Cayley and listed in OEIS (A000602) represents **acyclic constitutional (structural) isomers only**. Stereoisomers (chiral centers arising from branching) are excluded from these counts.

In [ ]:
# Alkane constitutional isomer counts (OEIS A000602)
alkane_isomers = [
    (1, "CH4", 1), (2, "C2H6", 1), (3, "C3H8", 1), (4, "C4H10", 2),
    (5, "C5H12", 3), (6, "C6H14", 5), (7, "C7H16", 9), (8, "C8H18", 18),
    (9, "C9H20", 35), (10, "C10H22", 75), (15, "C15H32", 4347),
    (20, "C20H42", 366319), (25, "C25H52", 36797588), (30, "C30H62", 4111846763),
    (40, "C40H82", 62481801147341)
]

df_iso = pd.DataFrame(alkane_isomers, columns=["Carbon_Count", "Formula", "Acyclic_Constitutional_Isomers"])
print("Acyclic alkane constitutional isomer growth table:")
df_iso.head(10)

### Enumeration of All 5 Hexane ($C_6H_{14}$) Constitutional Isomers

For $n=6$, exactly 5 constitutional isomers exist. Notice how molecular branching lowers the boiling point by reducing van der Waals surface contact area:

In [ ]:
# Hexane (C6H14) isomers
c6_isomers = [
    ("n-Hexane", "CCCCCC", 68.7),
    ("2-Methylpentane", "CCCC(C)C", 60.3),
    ("3-Methylpentane", "CCC(C)CC", 63.3),
    ("2,2-Dimethylbutane", "CCC(C)(C)C", 49.7),
    ("2,3-Dimethylbutane", "CC(C)C(C)C", 58.0),
]

c6_mols = [Chem.MolFromSmiles(smi) for _, smi, _ in c6_isomers]
c6_legends = [f"{name}\n(bp: {bp}°C)" for name, _, bp in c6_isomers]

# Draw the 5 constitutional isomers in a grid
img = Draw.MolsToGridImage(
    c6_mols,
    molsPerRow=5,
    subImgSize=(200, 150),
    legends=c6_legends
)
img

## 5. Functional Group SMARTS Library

A **functional group** is a specific arrangement of atoms within a molecule that imparts characteristic chemical reactivity, acid-base properties, and binding interactions.

In computational chemistry, functional groups are specified using **SMARTS (SMILES Arbitrary Target Specification)**.

`chemlib.descriptors.FUNCTIONAL_GROUPS_SMARTS` contains carefully designed patterns that avoid false positives (e.g., distinguishing basic amines from neutral amides using negative recursive SMARTS `!$(NC=O)`):

| Functional Group | SMARTS Pattern | Key Chemistry Notes |
|:---|:---|:---|
| **Alcohol** | `[OX2H]` | Aliphatic or aromatic hydroxyl |
| **Phenol** | `[OX2H][c]` | Hydroxyl attached directly to aromatic ring |
| **Ether** | `[OD2]([#6])[#6]` | Dialkyl/diaryl ether oxygen |
| **Aldehyde** | `[CX3H1](=O)[#6,H]` | Carbonyl with at least one hydrogen |
| **Ketone** | `[#6][CX3](=O)[#6]` | Carbonyl flanked by two carbons |
| **Carboxylic Acid** | `[CX3](=O)[OX2H1]` | Acidic carbonyl with hydroxyl |
| **Ester** | `[#6][CX3](=O)[OX2H0][#6]` | Carbonyl bonded to alkoxy oxygen |
| **Amide** | `[#6][CX3](=O)[NX3H0,NX3H1,NX3H2]` | Carbonyl bonded to nitrogen |
| **Primary Amine** | `[NX3H2;!$(NC=O)]` | $R-\text{NH}_2$ excluding amides |
| **Secondary Amine** | `[NX3H1;!$(NC=O)]([#6])[#6]` | $R_2\text{NH}$ excluding amides |
| **Tertiary Amine** | `[NX3H0;!$(NC=O)]([#6])([#6])[#6]` | $R_3\text{N}$ excluding amides |
| **Nitrile** | `[NX1]#[CX2]` | Cyano group ($-\text{C}\equiv\text{N}$) |
| **Nitro** | `[$([NX3](=O)=O),$([NX3+](=O)[O-])]` | Nitro group ($-\text{NO}_2$) |
| **Halogen** | `[F,Cl,Br,I]` | Fluorine, chlorine, bromine, iodine |

In [ ]:
# Demonstrate functional group counts on Aspirin and Paracetamol
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
paracetamol = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")

fg_asp = descriptors.compute_functional_group_counts(aspirin)
fg_para = descriptors.compute_functional_group_counts(paracetamol)

df_fg_demo = pd.DataFrame([fg_asp, fg_para], index=["Aspirin", "Paracetamol"]).T
# Filter only groups present in at least one molecule
df_fg_demo = df_fg_demo[(df_fg_demo > 0).any(axis=1)]
print("Detected functional groups:")
df_fg_demo

## 6. Aromaticity Perception: Hückel's Rule vs. RDKit Models

**Hückel's Rule** states that a planar, monocyclic, continuously conjugated ring system possesses special thermodynamic stability (aromaticity) if it contains **$4n+2$** $\pi$ electrons ($n = 0, 1, 2, \dots$).

Let's analyze the classic molecules discussed in lecture:
1. **Benzene**: 6 ring atoms, $6\pi$ electrons ($n=1$) $\rightarrow$ **Aromatic**.
2. **Cyclopentadienyl Anion ($C_5H_5^-$)**: 5 ring atoms, 4 from double bonds + 2 from carbanion lone pair = $6\pi$ electrons ($n=1$) $\rightarrow$ **Aromatic**.
3. **Tropylium Cation ($C_7H_7^+$)**: 7 ring atoms, 6 from double bonds + 0 from empty $p$-orbital = $6\pi$ electrons ($n=1$) $\rightarrow$ **Aromatic**.
4. **Cyclooctatetraene (COT)**: 8 ring atoms, $8\pi$ electrons ($4n, n=2$). Avoids anti-aromaticity by adopting a non-planar **tub conformation** $\rightarrow$ **Non-aromatic**.
5. **1-Methyl-2(1H)-pyridone**: Lactam with a polarized zwitterionic resonance contributor ($C^+-O^-$) that yields a $6\pi$ aromatic ring.
6. **Fulvene**: 5-membered ring with exocyclic double bond ($6\pi$ total, but cross-conjugated; dipole moment 1.1 D, non-aromatic ground state).
7. **Tropone (Cycloheptatrienone)**: Highly polarized $C=O$ ($C^+-O^-$) gives a $6\pi$ tropylium-like ring contributor.

In [ ]:
# Test molecules for aromaticity perception
aromatic_cases = [
    ("Benzene", "c1ccccc1", 6, True),
    ("Cyclopentadienyl anion", "[c-]1cccc1", 6, True),
    ("Tropylium cation", "[c+]1cccccc1", 6, True),
    ("Cyclooctatetraene", "C1=CC=CC=CC=C1", 8, False),
    ("Pyridine", "c1ccncc1", 6, True),
    ("Pyrrole", "c1cc[nH]c1", 6, True),
    ("Fulvene", "C1=CC=CC1=C", 6, False), # Exocyclic double bond
    ("Tropone", "O=C1C=CC=CC=C1", 6, False), # Partial aromatic character
]

results = []
for name, smi, pi_count, expected_aromatic in aromatic_cases:
    mol = Chem.MolFromSmiles(smi)
    rdkit_aromatic = False
    if mol is not None:
        # Molecule is considered aromatic by RDKit if any atom is aromatic
        rdkit_aromatic = any(atom.GetIsAromatic() for atom in mol.GetAtoms())
    
    # Hückel calculation for monocyclic ring: (pi - 2) % 4 == 0
    huckel_satisfied = (pi_count >= 2) and ((pi_count - 2) % 4 == 0)
    results.append({
        "Name": name,
        "SMILES": smi,
        "Pi_Electrons": pi_count,
        "Huckel_4n_plus_2": huckel_satisfied,
        "RDKit_Perceived": rdkit_aromatic
    })

df_arom = pd.DataFrame(results)
df_arom

## 7. Checkmol Overview and SMARTS Fallback

**Checkmol** (written by Norbert Haider, University of Vienna) is a specialized command-line utility written in Free Pascal that scans chemical structures against an internal dictionary of **204 functional groups** and outputs binary fingerprint codes (e.g. `checkmol -p file.mol` or `checkmol -b file.mol`).

In python environments where the external binary is not installed, our `chemlib.descriptors.FUNCTIONAL_GROUPS_SMARTS` module acts as a robust, fully native fallback that reproduces the functional group identification and fingerprint generation.

## 8. Functional Group Fingerprinting & Similarity

By taking the presence (1) or absence (0) of each functional group in our SMARTS library, we construct a **Functional Group Fingerprint (FG-FP)**.

Unlike path-based or circular fingerprints, an FG-FP is:
1. **Directly interpretable**: Each bit position corresponds to a specific chemical moiety (e.g., bit 0 = alcohol, bit 5 = carboxylic acid).
2. **Domain-focused**: Highly effective for QSAR tasks governed by specific pharmacophoric interactions (e.g., membrane transporters, efflux pumps).

In [ ]:
# Build functional group fingerprint function
def get_functional_group_fingerprint(mol: Chem.Mol) -> np.ndarray:
    """Return a binary 1D numpy array representing presence/absence of standard functional groups."""
    counts = descriptors.compute_functional_group_counts(mol)
    # Sort keys for consistent ordering
    keys = sorted(descriptors.FUNCTIONAL_GROUPS_SMARTS.keys())
    vector = np.array([1 if counts[k] > 0 else 0 for k in keys], dtype=int)
    return vector

# Compute FG fingerprints for a panel of 4 distinct drugs
drug_panel = [
    ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
    ("Ibuprofen", "CC(C)Cc1ccc(C(C)C(=O)O)cc1"),
    ("Paracetamol", "CC(=O)Nc1ccc(O)cc1"),
    ("Atenolol", "CC(C)NCC(O)COc1ccc(CC(N)=O)cc1"),
]

fg_matrix = []
keys = sorted(descriptors.FUNCTIONAL_GROUPS_SMARTS.keys())
for name, smi in drug_panel:
    m = Chem.MolFromSmiles(smi)
    fg_matrix.append(get_functional_group_fingerprint(m))

df_fg_matrix = pd.DataFrame(fg_matrix, index=[d[0] for d in drug_panel], columns=keys)
print("Functional Group Fingerprint Matrix (shape:", df_fg_matrix.shape, "):")
df_fg_matrix

### Case Study Preview: P-Glycoprotein Substrate Classification

P-glycoprotein (P-gp / ABCB1) is an ATP-dependent efflux pump that limits the brain penetration and bioavailability of therapeutic drugs. In Lecture 04, a famous study used ~250 functional-group bits to classify P-gp substrates and non-substrates with machine learning.

Let's load `data/processed/d07_pgp_classification.csv` and compare the functional group profiles of substrates vs. non-substrates.

In [ ]:
# Load P-gp classification dataset
pgp_file = repo_root / "data" / "processed" / "d07_pgp_classification.csv"
df_pgp = pd.read_csv(pgp_file)

print(f"Loaded {len(df_pgp)} drugs for P-gp analysis:")
print("Substrates (1):", (df_pgp["p_gp_substrate"] == 1).sum())
print("Non-substrates (0):", (df_pgp["p_gp_substrate"] == 0).sum())

# Compute average tertiary amine, ester, and ether counts
records = []
for idx, row in df_pgp.iterrows():
    m = Chem.MolFromSmiles(row["smiles"])
    counts = descriptors.compute_functional_group_counts(m)
    counts["p_gp_substrate"] = row["p_gp_substrate"]
    records.append(counts)

df_pgp_analysis = pd.DataFrame(records)
avg_by_class = df_pgp_analysis.groupby("p_gp_substrate")[["tertiary_amine", "ether", "ester", "aromatic_ring"]].mean()
print("\nAverage functional group counts by class:")
avg_by_class

## 9. Hands-On Exercises

### Exercise 7.1: Functional Group Identification for 5 Diverse Drugs
Given the 5 drugs below, compute their functional group counts and verify that:
1. **Aspirin** has both an ester and a carboxylic acid.
2. **Atenolol** contains an amide, an alcohol, an ether, and a secondary amine.

In [ ]:
drugs_test = {
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
    'Ibuprofen': 'CC(C)Cc1ccc(C(C)C(=O)O)cc1',
    'Atenolol': 'CC(C)NCC(O)COc1ccc(CC(N)=O)cc1',
    'Ciprofloxacin': 'O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O'
}

drug_counts = {}
for name, smi in drugs_test.items():
    m = Chem.MolFromSmiles(smi)
    drug_counts[name] = descriptors.compute_functional_group_counts(m)

print('Aspirin ester count:', drug_counts['Aspirin']['ester'])
print('Aspirin carboxylic acid count:', drug_counts['Aspirin']['carboxylic_acid'])
print('Atenolol amide count:', drug_counts['Atenolol']['amide'])

In [ ]:
# Self-check test for Exercise 7.1
assert drug_counts["Aspirin"]["ester"] >= 1, "Aspirin must have an ester"
assert drug_counts["Aspirin"]["carboxylic_acid"] >= 1, "Aspirin must have a carboxylic acid"
assert drug_counts["Atenolol"]["amide"] >= 1, "Atenolol must have an amide"
assert drug_counts["Atenolol"]["alcohol"] >= 1, "Atenolol must have an alcohol"
assert drug_counts["Ciprofloxacin"]["halogen"] >= 1, "Ciprofloxacin must have a fluorine halogen"
print("✔ Exercise 7.1 passed!")

### Exercise 7.2: Binary Functional Group Tanimoto Similarity
Implement a function `fg_tanimoto_similarity(mol1: Chem.Mol, mol2: Chem.Mol) -> float` that computes the Tanimoto similarity between the binary functional group vectors of two molecules:
$$T(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{a}{a + b + c}$$
where $a$ is the number of shared ON bits, and $b, c$ are unique ON bits in $A$ and $B$.

In [ ]:
def fg_tanimoto_similarity(mol1: Chem.Mol, mol2: Chem.Mol) -> float:
    # Complete working solution
    v1 = get_functional_group_fingerprint(mol1)
    v2 = get_functional_group_fingerprint(mol2)
    a = int(np.sum((v1 == 1) & (v2 == 1)))
    b = int(np.sum((v1 == 1) & (v2 == 0)))
    c = int(np.sum((v1 == 0) & (v2 == 1)))
    denom = a + b + c
    return float(a / denom) if denom > 0 else 0.0

m1 = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O') # Aspirin
m2 = Chem.MolFromSmiles('CC(=O)Nc1ccc(O)cc1')     # Paracetamol
print('FG Tanimoto between Aspirin and Paracetamol:', fg_tanimoto_similarity(m1, m2))

In [ ]:
# Self-check test for Exercise 7.2
m1 = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O") # Aspirin
m2 = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")     # Paracetamol
sim_self = fg_tanimoto_similarity(m1, m1)
sim_cross = fg_tanimoto_similarity(m1, m2)

assert np.isclose(sim_self, 1.0), f"Self similarity must be 1.0, got {sim_self}"
assert 0.0 < sim_cross < 1.0, f"Cross similarity should be between 0 and 1, got {sim_cross}"
print("✔ Exercise 7.2 passed!")

### Exercise 7.3: Hückel $(4n+2)$ Rule Validator
Implement `is_huckel_electron_count(n_pi: int) -> bool` that returns `True` if a given number of $\pi$ electrons satisfies the $4n+2$ rule for integer $n \ge 0$ ($2, 6, 10, 14, 18, \dots$).

In [ ]:
def is_huckel_electron_count(n_pi: int) -> bool:
    # Complete working solution
    if n_pi < 2:
        return False
    return (n_pi - 2) % 4 == 0

print('Is 6 pi Hückel?', is_huckel_electron_count(6))
print('Is 8 pi Hückel?', is_huckel_electron_count(8))

In [ ]:
# Self-check test for Exercise 7.3
assert is_huckel_electron_count(2) is True   # n=0 (e.g. cyclopropenyl cation)
assert is_huckel_electron_count(6) is True   # n=1 (benzene)
assert is_huckel_electron_count(10) is True  # n=2 (naphthalene)
assert is_huckel_electron_count(8) is False  # COT (antiaromatic/nonaromatic)
assert is_huckel_electron_count(4) is False  # cyclobutadiene
assert is_huckel_electron_count(0) is False
print("✔ Exercise 7.3 passed!")

## 10. Common Pitfalls

1. **Amide vs. Amine SMARTS Overlap**: A simple `[NX3]` query matches amides, anilines, and basic aliphatic amines indiscriminately. Because amides are neutral (pKa ~ -0.5) and amines are basic (pKa ~ 9–10), they behave completely differently in biological systems. Always use negative recursive constraints `!$(NC=O)`.
2. **Aromaticity Perception Models**: Different cheminformatics toolkits implement different aromaticity models (Hückel vs. RDKit vs. Daylight vs. MDL). For instance, nitro groups or cross-conjugated rings (fulvenes, tropones) may have their aromaticity perceived differently depending on whether formal charges or dative bonds are used.
3. **Stereochemistry Exclusion in Cayley Counts**: When quoting isomer counts for alkanes, remember that Cayley's formulas count *graph-theoretical trees* (constitutional isomers) and omit chiral stereocenters.

## 11. Summary & Key Takeaways

- **Carbon's Preeminence**: Hybridization ($sp^3, sp^2, sp$) and catenation yield an exponential diversity of organic architectures.
- **Isomer Diversity**: $C_6H_{14}$ has 5 constitutional isomers whose boiling points vary with degree of branching due to surface area effects.
- **SMARTS Pattern Design**: Functional group recognition requires precise SMARTS definitions to avoid false matches between related carbonyl and amino derivatives.
- **Hückel Aromaticity**: Systems containing $4n+2$ delocalized $\pi$ electrons in a planar monocyclic ring achieve distinct aromatic stability.
- **Interpretable Fingerprints**: Functional group presence/absence vectors enable transparent, chemically intuitive similarity search and machine learning classification.

## 12. References & Further Reading

- Cayley, A. On the mathematical theory of isomers. *Philos. Mag.* **1874**, 47, 444–446.
- Haider, N. Functionality pattern matching in substance databases: the `checkmol` utility. *Molecules* **2010**, 15, 5079–5092.
- Todeschini, R.; Consonni, V. *Molecular Descriptors for Chemoinformatics*. Wiley-VCH, **2009**.
- Didziapetris, R. et al. Classification analysis of P-glycoprotein substrate specificity. *J. Drug Target.* **2003**, 11, 391–406.

In [ ]:
# Final self-test cell
print("✔ NB07 checks passed")